# Feature Engineering & Selection

**Objective:** Rank features by importance and select optimal reduced feature set.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

plt.style.use('ggplot')
%matplotlib inline

In [ ]:
from src.data import DataLoader, DataCleaner, DataSplitter
from src.features import FeatureBuilder, FeatureSelector

# Load and clean
loader = DataLoader(config)
cleaner = DataCleaner(config)
splitter = DataSplitter(config)

train_df, test_df = loader.load_raw()
df = loader.merge_partitions(train_df, test_df)
df = cleaner.clean(df)

train_df, test_df = splitter.split_by_partition(df)
X_train, y_train, _ = cleaner.separate_features_target(train_df)
X_test, y_test, _ = cleaner.separate_features_target(test_df)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Rank features by importance
selector = FeatureSelector(config)
importance = selector.rank_features(X_train, y_train)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).plot(kind='barh', ax=ax)
ax.set_xlabel('Importance')
ax.set_title('Top 20 Features by Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_top20.png', dpi=150)
plt.show()

In [ ]:
# Evaluate reduced feature sets
results = []
for n in [10, 15, 20, 25, 30]:
    selected = selector.select_top_n(importance, n)
    metrics = selector.evaluate_reduced_set(X_train, y_train, X_test, y_test, selected)
    results.append(metrics)
    print(f"Top-{n:2d}: Acc={metrics['accuracy']:.4f}, Prec={metrics['precision']:.4f}, "
          f"Rec={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")

results_df = pd.DataFrame(results)
print(f"\nBest feature count: {results_df.loc[results_df['f1'].idxmax(), 'n_features']}")

In [ ]:
# Plot accuracy vs feature count
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(results_df['n_features'], results_df['accuracy'], 'o-', label='Accuracy')
ax.plot(results_df['n_features'], results_df['f1'], 's-', label='F1-Score')
ax.set_xlabel('Number of Features')
ax.set_ylabel('Score')
ax.set_title('Performance vs. Feature Count')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('../results/figures/feature_reduction_tradeoff.png', dpi=150)
plt.show()

In [ ]:
# Save selected features
best_n = int(results_df.loc[results_df['f1'].idxmax(), 'n_features'])
selected_features = selector.select_top_n(importance, best_n)
selector.save_selected_features(selected_features)
print(f"Selected {len(selected_features)} features:")
print(selected_features)